# Highbay Schema Evaluation - v5

Objective: Evaluate AST Syntax Compliance and Effect Parsing for v5 LoRA.

In [ ]:
# 1. Mount Google Drive & Create Missing Evaluation Directories
from google.colab import drive
import os
import json

drive.mount('/content/drive')

DRIVE_BASE = "/content/drive/MyDrive/HighbayGeniusTraining"
EVAL_REPORTS_DIR = os.path.join(DRIVE_BASE, "evaluation", "reports")
os.makedirs(EVAL_REPORTS_DIR, exist_ok=True)

In [ ]:
# 2. Setup & Imports
!pip install -q torch transformers unsloth evaluate metrics
from unsloth import FastLanguageModel
import torch

In [ ]:
# 3. Load Fine-Tuned Model & Adapter
# Check Colab's local filesystem first since it's faster, and fall back to Google Drive
RUN_ID = "v5-qwen2.5-1.5b"
LOCAL_MODEL_PATH = f"/content/adapter_{RUN_ID}"
DRIVE_MODEL_PATH = os.path.join(DRIVE_BASE, "training", "outputs", f"adapter_{RUN_ID}")

if os.path.exists(LOCAL_MODEL_PATH):
    MODEL_PATH = LOCAL_MODEL_PATH
    print(f"Loading adapter from local Colab filesystem: {MODEL_PATH}")
elif os.path.exists(DRIVE_MODEL_PATH):
    MODEL_PATH = DRIVE_MODEL_PATH
    print(f"Loading adapter from Google Drive: {MODEL_PATH}")
else:
    MODEL_PATH = LOCAL_MODEL_PATH
    print(f"Warning: Adapter path not found. Defaulting to local path: {MODEL_PATH}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_PATH,
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

In [ ]:
# 4. Test Corpus
test_cases = [
    {
        "category": "ambiguous_relation",
        "prompt": "Add a user profile screen where a user is linked to a person and has a favorite color."
    },
    {
        "category": "effect_primitive",
        "prompt": "Create a checkout review step that displays user().email and cart().total."
    },
    {
        "category": "more_of_objects",
        "prompt": "Add a repeating section where users can add multiple shipping addresses with street and zip code."
    }
]

PROMPT_TEMPLATE = """Below is an instruction that describes a UI design action or natural language prompt. Write the corresponding Typed Markdown Intermediate Representation (IR).

### Instruction:
{prompt}

### Typed Markdown IR:
"""

In [ ]:
# 5. Execution & Inference Loop
results = []
for case in test_cases:
    prompt_text = PROMPT_TEMPLATE.format(prompt=case["prompt"])
    inputs = tokenizer([prompt_text], return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        **inputs, 
        max_new_tokens = 512, 
        temperature = 0.1, 
        use_cache = True
    )
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    generated_ir = decoded.split("### Typed Markdown IR:")[-1].strip()
    
    results.append({
        "category": case["category"],
        "prompt": case["prompt"],
        "generated_ir": generated_ir
    })

In [ ]:
# 6. Validation Metrics & Report Export to Drive
def evaluate_ir_syntax(ir_text):
    metrics = {
        "has_frontmatter": "---" in ir_text,
        "has_glyphs": any(g in ir_text for g in ["{{ abc", "{{ #", "{{ $", "{{ ⭘", "{{ ⚯"]),
        "has_effects": any(e in ir_text for e in ["user()", "cart()", "checkoutSession()"]),
        "valid_markdown_list": "- [" in ir_text
    }
    score = sum(metrics.values()) / len(metrics)
    return score, metrics

print("=== EVALUATION RESULTS (RUN v5) ===")
report_data = []
for r in results:
    score, metrics = evaluate_ir_syntax(r["generated_ir"])
    report_item = {
        "category": r["category"],
        "prompt": r["prompt"],
        "score": score,
        "metrics": metrics,
        "generated_ir": r["generated_ir"]
    }
    report_data.append(report_item)
    print(f"\nCategory: {r['category']}")
    print(f"Compliance Score: {score * 100}%")
    print(f"Metrics: {metrics}")
    print(f"Generated Output:\n{r['generated_ir']}\n" + "-"*40)

report_file_path = os.path.join(EVAL_REPORTS_DIR, "eval_report_v5.json")
with open(report_file_path, "w") as f:
    json.dump(report_data, f, indent=2)
print(f"Evaluation report saved to {report_file_path}")